<a href="https://colab.research.google.com/github/frankausberlin/deen-lupysta/blob/main/scripts/deendash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table width='1300'><tr align=center><td><a href='https://suno.com/playlist/dea7acde-5886-407a-8193-db889a2af569'><table width=800><tr><td>

![][deenpic]

[deenpic]: https://lh3.googleusercontent.com/d/1JdavIkcaCKkXrqiJtGHpAMvXAp73Z8tA "Linux Punk Songs"
</td></tr></table></a></td></tr></table>


## Data Science Dash

* This notebook contains a few useful snippets on the subject of data science.
* The snippets use Colab forms to hide the code and display a description.
* Additionally, basic settings can optionally be provided via the Colab forms mechanism (`#@param`).
* In addition to the data science snippets, there are also basic widget snippets—ButtonBox and Selectable—which are used by the others.



### Base Widgets

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**ButtonBox**</font><br>
#@markdown * An alternative for the Tab widget for selecting **large amount of items**
#@markdown * Uses **Buttons** and a **Box** with **flex wrap layout**.
#@markdown * Provide a **mechanism for select, append and delete** an item .
#@markdown <table><tr><td><font size='+2'>
#@markdown
#@markdown ```python
#@markdown def showSelection (bbox):
#@markdown     try:    textarea.value = plain (render_doc(bbox.button.tooltip))
#@markdown     except: textarea.value = 'error calling render_doc with '+bbox.button.tooltip
#@markdown #
#@markdown textarea  = Textarea  (layout=Layout(width='1000px', height='300px'))
#@markdown buttonbox = ButtonBox (__builtins__.__dict__.keys(), clicker=showSelection)
#@markdown display (buttonbox.box, textarea)
#@markdown ```
from ipywidgets import Button, Box, Layout, Textarea

class ButtonBox ():
  def __init__ (self, descriptions, clicker=None, maxchar=60, color=None):
    # remember stuff - list to set
    self.descriptions = descriptions if len (descriptions) == len (set (descriptions)) else set (descriptions)
    self.clicker, self.maxchar, self.color, self.position, self.button = clicker, maxchar, color, -1, None
    self._original_colors = {} # Dictionary to store original colors

    # make buttons
    self.buttons = [Button (description = i if len (i) <= maxchar else f'{i[:(maxchar-3)]}...',
                            layout      = Layout(width='auto', height='22px'),
                            tooltip     = f'{i}')
                    for i in descriptions]

    # put them in a box / bind event
    self.widget = Box (layout   = Layout (display='flex', flex_flow='wrap'),
                       children = self.buttons)
    for button in self.buttons:
      self._original_colors[button] = button.style.button_color # Store original color
      button.on_click (self._clicker)


  def color_buttons (self, colors):
    # new colors for buttons
    for b,c in zip(self.buttons[:len(colors)], colors):
      b.style = {'button_color': c}
      self._original_colors[b] = c


  # An alternative for the Tab widget
  def _clicker (self, b):
    # search clicked button and remember
    self.position, self.button = [(i,but) for i, but in enumerate(self.buttons) if b == but][0]

    # unselect (color) all buttons and select new (list of colors here: https://www.quackit.com/css/css_color_codes.cfm)
    self.unselect()
    if self.color: self.buttons[self.position].style={'button_color':self.color}
    self.buttons[self.position].layout.border = '1px solid black'

    # fire event
    if self.clicker: self.clicker (self)


  def append (self, description, select=True):
    # add new selctor button at the end
    b = Button (description = description if len (description) <= self.maxchar else f'{description[:(self.maxchar-3)]}...',
                layout      = Layout(width='auto', height='21px'),
                tooltip     = f'{description}')
    self.buttons.append(b)
    self._original_colors[b] = b.style.button_color # Store original color for the new button
    self.widget.children = [*self.widget.children, b]

    # select new button / bind event
    if select:
      self.button, self.position = b, len(self.buttons) - 1
      self._clicker (b)
    b.on_click (self._clicker)


  def remove (self, position=None):
    # remove selected if no position given
    if not position: position = self.position
    if not position or not position < len(self.buttons) or position < 0: return

    # clean up
    removed_button = self.buttons[position]
    del self._original_colors[removed_button] # Remove from original colors dictionary
    self.widget.children = [*self.widget.children[:position], *self.widget.children[position+1:]]
    self.buttons = [*self.buttons[:position], *self.buttons[position+1:]]
    self.button, self.position = None, -1


  def unselect (self):
    # unselect all buttons and restore original color
    for b in self.buttons:
      b.layout.border = ''
      b.style = {'button_color': self._original_colors.get(b, None)} # Restore original color, default to None if not found

# ___________________________________________________________
#|______________________hello_component______________________|
from ipywidgets   import Layout, Textarea, Button
from pydoc        import plain, render_doc

# simple button box example that shows the python builtins
def showSelection (bbox):
  try:    textarea.value = plain (render_doc(bbox.button.tooltip))
  except: textarea.value = 'error calling render_doc with '+bbox.button.tooltip # usage tooltip (description may be shortened)
buttonbox = ButtonBox (__builtins__.__dict__.keys(), clicker=showSelection, color='powderblue')

# some colors for the first 11 buttons
buttonbox.color_buttons ([*['MediumAquamarine']*3, *['Violet']*3, *['Moccasin']*5])

# area for the doc-string
textarea  = Textarea  (layout=Layout(width='1000px', height='300px'))

# add dummy button at the end / delete selected button
(del_but := Button (description="delete")).on_click (lambda b: buttonbox.remove ())
(app_but := Button (description="append")).on_click (lambda b: buttonbox.append ('new button'))

# display or not
# You can prevent widgets from being displayed by setting DISPLAY_WIDGETS to "False."
# The idea behind this is to allow ipywidgets to be used outside of an iPython environment
# (without the display command). The ipywidgets work even without being rendered.
try:
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (buttonbox.widget, textarea)
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (app_but, del_but)
except: pass


In [ ]:
""" Colab Code-Snippets
It uses colab forms to make a top-level setting of the code snippet (optional)
and give a little description. You can run the cell to execute the code snippet.
"""
#@markdown <font size='+2' color='#005F6A'>**Selectable**</font><br>
#@markdown * Provides a mechanism for selecting items with either **radio**, **multi-select** or **radiox** (unselecteble radio) behavior.
#@markdown * Uses **Buttons** to represent selectable items.
#@markdown * Allows for **custom behavior** on first selection and subsequent selections.
#@markdown <table><tr><td><font size='+2'>
#@markdown
#@markdown ```python
#@markdown class MySelectable (Selectable):
#@markdown   def setInitState (self):                           self.setItemWidget (Label (value="initial state"))
#@markdown   def onItemSelect (self, posList, lastSelection):   print ('Your selection:',posList, 'Last selection:',lastSelection)
#@markdown   def __init__(self, items, behave='radio'):
#@markdown     super().__init__(item, items, behave=behave, selector=self.onItemSelect)
#@markdown #
#@markdown lst = []
#@markdown display (HBox (children = [MySelectable(lst, behave='multi').widget,
#@markdown                            MySelectable(lst, behave='multi').widget]) )
#@markdown ```
#@markdown </td></tr></table>
from ipywidgets import Button, HTML, HBox, VBox, Label, Output
from IPython.display import display

class Selectable:
  """A class representing a selectable item with customizable behavior.
  This code snippet demonstrates the use of a custom selectable widget class
  in a Jupyter notebook. It allows for the selection of items using either
  radio button behavior or multi-select behavior.

  Attributes
  ----------
  * items: list
    A list with Selectable objects.
  * behave: str
    The behavior type ('multi' or 'radio' or 'radiox') for selection.
  * onSelect: function | default: None
    A callback function triggered upon selection.
  * widget: HBox
    The widget representing the selectable item.

  Methods
  -------
  * doBehavior() | -> none
    Executes the selection behavior based on the specified type.
  * _onSelectorClick(b: Button) | -> none
    Handles the click event for the selector button.
  * setItemWidget(widget) | -> none
    Sets the widget to display for the item.
  * setInitState() | -> none
    Initializes the state of the item (to be implemented in subclasses).
  * onFirstSelect(posList) | -> none
    Defines behavior on the first selection (to be implemented in subclasses).

  Behavior
  -------
  * You can set the behavior to 'radio' or 'multi' (radio button or multi selection).
  * On creation it calls the setInitState abstract function where you can build your
    own widget (e.g. HTML-object) and register it with setItemWidget.
  * When you select an item the abstract function onFirstSelect is called so you can
    set your widget in a 'working' state.
  * Every selection triggers the callback function onSelect if set.

  Use cases
  ---------
  * Make your own class, inherit from Selectable and implement the onFirstSelect
    and setInitState function.
  * To react on a select you can give a listener in the super constructor call.
  * To combine several selectable objects to a set you have to give the same list
    on construction.
  * Display an item with the .widget attribute.

  Example
  -------
  class MySelectable (Selectable):
    def setInitState (self):            self.setItemWidget (Label (value="initial state"))
    def onFirstSelect (self, posList):  self.itemWidget.value = "initialized "+self.item
    def onSelection (self, posList):    print ('Your selection:',posList)
    def __init__(self, items, behave='radio'):
      super().__init__(items, behave, self.onSelection)

  # example: two sets of Selectable objects (two lines radio button / one line multi select)
  a, b = [] , []
  #
  display (HBox (children = [MySelectable(a).widget, MySelectable(a).widget, MySelectable(a).widget] ))
  display (HBox (children = [MySelectable(a).widget, MySelectable(a).widget, MySelectable(a).widget] ))
  display (HTML(value='<hr>'))
  #
  display (HBox (children = [MySelectable(b, 'multi').widget, MySelectable(b, 'multi').widget,
                             MySelectable(b, 'multi').widget, MySelectable(b, 'multi').widget,
                             MySelectable(b, 'multi').widget, MySelectable(b, 'multi').widget] ))

  """
  def __init__(self, items, behave, selector=None):
    """Initializes the class with given parameters.
    set attributes, generate widgets, bind events and set initial state.
    """
    self.isSelected, self.items, self.behave = False, items, behave
    self.bu_selector = Button(style={'button_color': '#99bfc3'}, layout={'width': '22px', 'height': '22px'})
    self.widget = HBox(children=[self.bu_selector], layout={'min_height': '24px', 'overflow': 'hidden'})
    self.items.append(self)
    self.bu_selector.on_click(self.select)
    self.setInitState()
    self.selector = selector
    self.posList = []


  def doBehavior (self):
    """Executes the selection behavior based on the specified type.
    """
    self.posList, lastSelect = [], self.lastSelect()

    if self.behave == 'radiox':
      state         = self.items[lastSelect].isSelected
      for item in self.items: item.bu_selector.style.button_color, item.isSelected = '#99bfc3', False
      if state:     self.items[lastSelect].bu_selector.style.button_color, self.items[lastSelect].isSelected = '#99bfc3', False
      else:         self.items[lastSelect].bu_selector.style.button_color, self.items[lastSelect].isSelected = '#005F6A', True
      self.posList  = [p for p in range (len(self.items)) if self.items[p].isSelected]

    elif self.behave == 'radio':
      for item in self.items: item.bu_selector.style.button_color, item.isSelected = '#99bfc3', False
      self.bu_selector.style.button_color, self.isSelected, self.posList = '#005F6A', True, [lastSelect]

    elif self.behave == 'multi':
      state         = self.items[lastSelect].isSelected
      if state:     self.items[lastSelect].bu_selector.style.button_color, self.items[lastSelect].isSelected = '#99bfc3', False
      else:         self.items[lastSelect].bu_selector.style.button_color, self.items[lastSelect].isSelected = '#005F6A', True
      self.posList  = [p for p in range (len(self.items)) if self.items[p].isSelected]

    else: raise Exception('unknown behavior: '+self.behave)

  def setItemWidget (self, widget, tooltip=None):
    """Sets the widget to display for the item.
    """
    self.itemWidget, self.widget.children = widget, (*self.widget.children, widget)
    if tooltip: self.bu_selector.tooltip = tooltip

  def setInitState (self): raise NotImplementedError
  def onSelection (self, posList): raise NotImplementedError

  def select (self, b=None):
    """Handles the click event for the selector button.
    """
    lastSelect=-1
    # search selected item
    for buttonPos in range (len(self.items)):
      if self.items[buttonPos].bu_selector == self.bu_selector: break
    # when found
    if buttonPos < len(self.items):
      # set isLastSelect in all selectables
      for buttonPos in range (len(self.items)): self.items[buttonPos].isLastSelect = False
      self.isLastSelect = True
    # do selectable specific behavior
    self.doBehavior ()
    # call selector
    if self.selector:
      self.selector (self.posList, self.lastSelect())

  def lastSelect(self):
    for i, item in enumerate(self.items):
      if item.bu_selector == self.bu_selector: return i
    return -1

  def setSelection (self, posList):
    for buttonPos in range (len(self.items)):
      if self.items[buttonPos].bu_selector == self.bu_selector: break

# ___________________________________________________________
#|______________________hello_component______________________|
output = Output ()
class HelloSelectable (Selectable):
  def setInitState (self): self.setItemWidget (Label (value="initial state"))
  def onItemSelect (self, posList, lastSelect=None):
    output.clear_output()
    with output: print ('Your selection:',posList,'- lastSelection',lastSelect, end='')
  def __init__(self, items, behave='radio'):
    super().__init__(items, behave, self.onItemSelect)

# example: 3 sets of Selectable objects (radio button / multi select / radiox)
a, b, c = [] , [], []

# display or not
# You can prevent widgets from being displayed by setting DISPLAY_WIDGETS to "False."
# The idea behind this is to allow ipywidgets to be used outside of an iPython environment
# (without the display command). The ipywidgets work even without being rendered.
try:
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (HTML(value='<font size=+1>behave: radio'))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS:
    display (HBox (children = [HelloSelectable(a).widget, HelloSelectable(a).widget,
                               HelloSelectable(a).widget, HelloSelectable(a).widget,
                               HelloSelectable(a).widget, HelloSelectable(a).widget] ))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (HTML(value='<hr><font size=+1>behave: multi'))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS:
    display (HBox (children = [HelloSelectable(b, 'multi').widget, HelloSelectable(b, 'multi').widget,
                               HelloSelectable(b, 'multi').widget, HelloSelectable(b, 'multi').widget,
                               HelloSelectable(b, 'multi').widget, HelloSelectable(b, 'multi').widget] ))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (HTML(value='<hr><font size=+1>behave: radiox'))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS:
    display (HBox (children = [HelloSelectable(c, 'radiox').widget, HelloSelectable(c, 'radiox').widget,
                               HelloSelectable(c, 'radiox').widget, HelloSelectable(c, 'radiox').widget,
                               HelloSelectable(c, 'radiox').widget, HelloSelectable(c, 'radiox').widget] ))
  if not 'DISPLAY_WIDGETS' in globals() or DISPLAY_WIDGETS: display (output)
except: pass

### Data Science Snippets

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**DSCudaOnHostTest**</font><br>Check if cuda runing on host and is available for **pytorch, tensorflow, jax** and **numba**.

import os, warnings

# if you are using > Python 3.11:
# with warnings.catch_warnings(action="ignore"):

with warnings.catch_warnings():
  warnings.simplefilter("ignore")

  try:
    import torch
    cuda_pytorch = torch.cuda.is_available()
  except: cuda_pytorch = False

  try:
    import tensorflow as tf
    cuda_tensorflow = ':GPU:' in str(tf.config.list_physical_devices('GPU'))
  except: cuda_tensorflow = False

  try:
    import jax
    cuda_jax = len(jax.devices()) > 0
  except: cuda_jax = False

  try:
    from numba import cuda
    cuda_numba = cuda.is_available()
  except: cuda_numba = False

  try:
    from IPython.display import clear_output
    clear_output ()
  except: pass


  hasNvidiaSmi  = os.system ('nvidia-smi --version >/dev/null 2>&1') == 0
  if hasNvidiaSmi:
    print (f"pytorch-cuda:     \x1b[92mTrue\x1b[0m") if cuda_pytorch     else print (f"pytorch-cuda:     \x1b[91mFalse\x1b[0m")
    print (f"tensorflow-cuda:  \x1b[92mTrue\x1b[0m") if cuda_tensorflow  else print (f"tensorflow-cuda:  \x1b[91mFalse\x1b[0m")
    print (f"jax-cuda:         \x1b[92mTrue\x1b[0m") if cuda_jax         else print (f"jax-cuda:         \x1b[91mFalse\x1b[0m")
    print (f"numba-cuda:       \x1b[92mTrue\x1b[0m\n") if cuda_numba       else print (f"numba-cuda:       \x1b[91mFalse\x1b[0m")
    !nvidia-smi

  else:
    print ('\x1b[91mno cuda on host\x1b[0m')


In [ ]:
#@markdown <font size='+2' color='#005F6A'>**DSOpenRouterModels**</font><br>
#@markdown * Uses **openrouter-cli**-pip to get model infos from OpenRouter. It **must be installed**: `uv tool install openrouter-cli` or `pip install openrouter-cli`
#@markdown * You must insert your **OpenRouter API-key** first with: `openrouter-cli configure`'.
#@markdown * You can **specify** a comma-separated **list of frontier models** to be **scanned**, whose **latest versions** will be displayed in the **overview**.
from ipywidgets import Button, Box, Layout, Textarea, VBox, HTML, HBox
import subprocess, json, pprint

class ORMButtonBox ():
  def __init__ (self, descriptions, clicker=None, maxchar=60, color=None):
    # remember stuff
    self.descriptions, self._original_colors = set (descriptions), {}
    self.clicker, self.maxchar, self.color, self.position, self.button = clicker, maxchar, color, -1, None

    # make buttons
    self.buttons = [Button (description = i if len (i) <= maxchar else f'{i[:(maxchar-3)]}...',
                            layout      = Layout(width='auto', height='22px'),
                            tooltip     = f'{i}')
                    for i in descriptions]

    # put them in a box / bind event
    self.widget = Box (layout = Layout (display='flex', flex_flow='wrap'), children = self.buttons)
    for button in self.buttons:
      self._original_colors[button] = button.style.button_color # Store original color
      button.on_click (self._clicker)

  # An alternative for the Tab widget
  def _clicker (self, b):
    # search clicked button and remember
    self.position, self.button = [(i,but) for i, but in enumerate(self.buttons) if b == but][0]

    # unselect (color) all buttons and select new (list of colors here: https://www.quackit.com/css/css_color_codes.cfm)
    self.unselect()
    if self.color: self.buttons[self.position].style={'button_color':self.color}
    self.buttons[self.position].layout.border = '1px solid black'

    # fire event
    if self.clicker: self.clicker (self)

  def unselect (self):
    # unselect all buttons and restore original color
    for b in self.buttons:
      b.layout.border = ''
      b.style = {'button_color': self._original_colors.get(b, None)} # Restore original color, default to None if not found

def selectOrganization (bbox):
  global selectedOrga
  selectedOrga =  bbox.button.tooltip
  org_models = [m for m in json_list if m['id'].split('/')[0] == selectedOrga]
  modelsBox = ORMButtonBox ([i['id'].replace (bbox.button.tooltip+'/','') for i in org_models], clicker=selectModel, color='powderblue')
  widgetList.children=(widgetList.children[0], widgetList.children[1], modelsBox.widget)

def selectModel (bbox):
  for model in json_list:
    if model['id'] == selectedOrga+'/'+bbox.button.tooltip:
      if len (widgetList.children) < 4:
        widgetList.children = (*widgetList.children, Textarea (layout=Layout(width='auto', height='500px')))
      widgetList.children[3].value = pprint.pformat(model, width=160)

# ___________________________________________________________
#|______________________hello_component______________________|
try:
  json_list = json.loads(subprocess.run(['openrouter-cli','models', '--raw'], capture_output=True, text=True, check=True).stdout)
  tab_list = subprocess.run(['openrouter-cli','models'], capture_output=True, text=True, check=True).stdout.split('\n')
except: json_list = []

if json_list:

  # generate frontier models overview
  frontier_models = "gpt-5, claude-opus-4, claude-sonnet-4, gemini-3, deepseek-v4, minimax-m2, kimi-k2, glm-5, qwen3, mimo-v2, grok-4, grok-build-0"  #@param {type:"string"}
  ignore_frontier_models_with = "8b, 27b, 35b, a3b, -lite, -image, -micro, -nano, qwen3.6-flash"  #@param {type:"string"}
  ignore = ignore_frontier_models_with.replace(' ', '').split(',')

  # build the list with tuples ('versionnr', 'model-id') for the frontier models
  frontiers = frontier_models.replace(' ', '').split(',')

  # correct filter
  bad_id = lambda id: sum([f in id for f in ignore])

  frontier_version_id_list = [(j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0] , j['id'])
                              for f in frontiers for j in json_list
                                if f in j['id'] and
                                  ':free' not in j['id'] and
                                  j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0][-1] in '0123456789' and
                                  not bad_id (j['id'])]

  # build the frontier model overview
  tmp = []
  for frontier in frontiers:
    match = sorted ([f for f in frontier_version_id_list if frontier in f[1]])
    if not match: continue # catch empty lists due to rigorous filtering

    show_version = match[-1][0]
    versions = [f for f in match if show_version == f[0]]
    hm = f"<font color=blue size=3><b>{versions[0][1].split('/')[0]}</b></font><hr>"
    for version in versions:
      hm += f"<b>{version[1].split('/')[1]}</b><br>"
      idj = [j for j in json_list if j['id'] == version[1]][0]
      r, w  = f"{(float (idj['pricing']['prompt'])*1000000):.2f}", f"{(float (idj['pricing']['completion'])*1000000):.2f}"
      cr = f"{(float(idj['pricing']['input_cache_read'])*1000000):.2f}" if 'input_cache_read' in idj['pricing'] else ' '
      cw = f"{(float(idj['pricing']['input_cache_write'])*1000000):.2f}" if 'input_cache_write' in idj['pricing'] else ' '
      hm += f"Context: {idj['context_length']}<br>Price: {r} / {w}<br>[Cache: {cr} / {cw}]<br>"

    tmp.append (HTML(value=hm[:-4], layout={"border":"2px solid yellow"}))
  overview_panel = HBox (children=tmp)

  # build orga / model selector
  selectedOrga      = None
  organization_list = sorted (set ([j['id'].split('/')[0] for j in json_list if '/' in j['id']]))
  organizationsBox  = ORMButtonBox (organization_list, clicker=selectOrganization, color='powderblue')
  title             = HTML (value="<font color=green size=4><b>Overview Frontier Models Latest")
  widgetList        = VBox (children=[organizationsBox.widget,HTML(value='<hr>')])

  display (VBox (children=[title,overview_panel]), widgetList)

**Cognitive baseline test for selfhosted LLMs**
<table><tr><td>

```shell
Here is a classic math problem. Peter and Hans are talking.
Peter says: “If you give me one of your apples, then we'll both have the same number of apples.”
Hans replies: “But if you give me one of your apples, then I will have twice as many apples as you.”
How many apples does Peter have and how many apples does Hans have?
```

```shell
Answer: Peter 5 and Hans 7
```
</td></tr></table>

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**DSOllamaTools**</font><br>
#@markdown A little **IPyWidgets-GUI** to make **ollama calls with tools**.
#@markdown * A **selection option** is provided for **Ollama models** and **passed tools**, as well as a **text input** field for a **prompt**.
#@markdown * **Scans** for functions starts with **'tool_'** and **offers** them as **ollama** tools.
#@markdown * You can simple **write your own:**<br>
#@markdown   Create a new **cell with a function** whose name **starts with 'tool_'** and run the cell.
#@markdown      <table><tr><td>
#@markdown
#@markdown ```python
#@markdown def tool_get_server_status (server_name: str) -> str:
#@markdown   """Returns the current status of a specific server."""
#@markdown   if server_name.lower() == "orangepi": return f"{server_name} ok"
#@markdown   return f"{server_name} failed"
#@markdown ```
#@markdown ⚠️ Make sure that a <b>meaningful description</b> is present (within the triple quotes).
#@markdown </td></tr></table>
#@markdown
#@markdown Afterwards, **run the DSOllamaTools cell again, and the tool will appear** in the selection.
import types, inspect, ollama, random
from ipywidgets import Dropdown, Textarea, Output, HTML, Button, HBox, Label
from IPython.display import Markdown, display

# ollama config
ollama_seed = '4222' # @param {"type":"string","placeholder":"random"}
ollama_num_ctx = 4096 # @param {"type":"integer","placeholder":"e.g. 4096"}
ollama_num_predict = 3000 # @param {"type":"integer","placeholder":"e.g. 2048"}
ollama_temperatur = 0.3 #@param {type:"slider", min:0, max:1, step:0.1}
ollama_top_p = 0.4 #@param {type:"slider", min:0, max:1, step:0.1}
ollama_repeat_penalty = 1 #@param {type:"slider", min:1, max:2, step:0.1}
ollama_seed_int = int (ollama_seed) if ollama_seed and int (ollama_seed) else random.randint(0, 2**32-1)

# sample tool functions
def tool_get_city_weather (city_name: str) -> str:
  """Returns the current weather informations of a specific city."""
  if city_name.lower() == 'berlin': return "27 degrees, sunny, light wind from the northeast"
  else: return "7 degrees, drizzle, strong wind with gusts from the southwest"
def tool_get_server_status (server_name: str) -> str:
  """Returns the current status of a specific server."""
  if server_name.lower() == "orangepi": return f"{server_name} ok"
  return f"{server_name} failed"

try:
  # Extract the global tools
  global_tools = {name: obj for name, obj in globals().items() if isinstance(obj, types.FunctionType) and 'tool_' in name}

  # the tool selector based on Selectable
  class ToolSelectable(Selectable):
    def __init__(self, func_name, items, behave='multi'):
      # Store the function name before the `super()` call so that `setInitState()` can use it directly.
      self.func_name   = func_name

      # Tooltip infos
      self.signature   = inspect.signature (global_tools[func_name])
      self.description = inspect.getdoc (global_tools[func_name])

      # super call
      super().__init__(items, behave=behave, selector=self.onItemSelect)

    def setInitState(self):
      # Sets the function name as a label next to the button with the tooltip.
      tt = f'{self.func_name}\n\nSigniture: {self.signature}\nDescription: {self.description}'
      self.setItemWidget(Label(value=self.func_name), tooltip=tt)

    # is expected by the base class
    def onItemSelect (self, posList, lastSelect=None): pass

  # Create Selectables for all tools
  tools_list = [] # Empty list for the selectables
  for k in global_tools.keys(): ToolSelectable (k,tools_list) # They insert themselves into the list.

  # Dropdown with ollama models
  ollama_model = Dropdown (options=[m.model for m in ollama.list().models])

  # textfield prompt
  prompt = Textarea (value='Could you check how the OrangePi server is doing?\nAnd please also check what the weather is like in Berlin.',
                     layout={'width':'600px', 'height':'120px'})

  # a little wait animation
  wait_animation = HTML (value='', layout={'width':'20px', 'height':'20px'})

  # send button and send function
  send = Button (description='Send', layout={'width':'100px'})
  def send_ollama (b):
    # waiting on
    send.disabled  = True
    wait_animation.value = "<marquee width=20><b>&lt;&nbsp;&nbsp;&nbsp;&nbsp;&lt;</marquee>"

    # create the message stack with the first message and update markdown head
    collected_content, messages, markdown_head = '<hr>', [{'role': 'user', 'content': prompt.value}], f'## {ollama_model.value}\n'
    display_handle.update(Markdown(markdown_head))

    try:
      # start stream
      active_tools     = [global_tools[selectable.func_name] for selectable in tools_list if selectable.isSelected]
      ollama_options   = {'temperature':    ollama_temperatur,
                          'num_ctx':        ollama_num_ctx,
                          'num_predict':    ollama_num_predict,
                          'seed':           ollama_seed_int,
                          'repeat_penalty': ollama_repeat_penalty,
                          'top_p':          ollama_top_p}
      response_stream  = ollama.chat (model     = ollama_model.value,
                                      messages  = messages,
                                      stream    = True,
                                      tools     = active_tools,
                                      options   = ollama_options)

      # response chunk loop
      for chunk in response_stream:

        # content chunks
        if (msg := chunk.get('message', {})).get('content'):
            collected_content += msg['content'] # collect content
            display_handle.update(Markdown(markdown_head+collected_content))

        # thinking chunks
        if msg.get('thinking'):
            collected_content += msg['thinking'] # collect thinking
            display_handle.update(Markdown(markdown_head+collected_content))

        # tool call chunks
        if msg.get('tool_calls'):

          # add msg with toolcalls to the message stack
          messages.append(msg)

          # loops the tool calls
          for tool_call in msg['tool_calls']:

            # get function name and arguments from the tool_call list and update markdown_head
            func_name, arguments = tool_call.function.name, tool_call.function.arguments
            markdown_head += f"* Agent try execute: `{func_name}({arguments})`"

            # Run the function localy in Python, append result as a new message with the role 'tool' and update markdown head
            if func_name in global_tools:
              try:
                messages.append ({'role': 'tool', 'name': func_name, 'content': global_tools[func_name](**arguments)})
                markdown_head += ' ✅\n'
              except Exception as e:
                messages.append ({'role': 'tool', 'name': func_name, 'content': f"Error: {e}"})
                markdown_head += f' ❌ Error: {e}\n'
            else:
              messages.append ({'role': 'tool', 'name': func_name, 'content': f"Error: Function {func_name} not found."})
              markdown_head += f"❌ not found\n"

            # show tool infos in markdown head
            display_handle.update(Markdown(markdown_head+collected_content))

          # send new message stack with results of tool calls to ollama
          new_response_stream = ollama.chat(model=ollama_model.value, messages=messages, stream=True, tools=active_tools)

          # tool-call-results response chunk loop
          for new_chunk in new_response_stream:

            # add content of new response stream to old collected content
            if (nmsg := new_chunk.get('message', {})).get('content'):
              collected_content += nmsg['content']

            # and show it
            display_handle.update(Markdown(markdown_head+collected_content))

    # Perhaps the model does not support tool calls, or something else went wrong.
    except Exception as e:
      display_handle.update(Markdown(f'Error: {e}\n\n**try again with other model**'))

    # waiting off
    send.disabled, wait_animation.value  = False, ''

  send.on_click (send_ollama)

  # display the UI (title html, model dropdown, tool selector, prompt textarea, send button)
  display (HTML(value='<h3>Select Ollama Model</h3>'), ollama_model, HTML(value='<h3>Select Tools</h3>'))
  display (HBox(children=[t.widget for t in tools_list], layout={'flex_wrap': 'wrap'}))
  display (HTML(value='<h3>Prompt</h3>'), prompt, HBox(children=[send, wait_animation]))
  display_handle = display(Markdown(""), display_id=True)

except NameError as e:
  print ('Something extraordinarily stupid happened:\n\n', e)